In [1]:
import pandas as pd

df = pd.read_csv("winemag-data-130k-v2.csv")

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   id                     129971 non-null  int64  
 1   country                129908 non-null  object 
 2   description            129971 non-null  object 
 3   designation            92506 non-null   object 
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  object 
 7   region_1               108724 non-null  object 
 8   region_2               50511 non-null   object 
 9   taster_name            103727 non-null  object 
 10  taster_twitter_handle  98758 non-null   object 
 11  title                  129971 non-null  object 
 12  variety                129970 non-null  object 
 13  winery                 129971 non-null  object 
dtypes: float64(1), int64(2), object(11)


In [3]:
(len(df) - df.count(0)) / len(df) * 100

id                        0.000000
country                   0.048472
description               0.000000
designation              28.825661
points                    0.000000
price                     6.921544
province                  0.048472
region_1                 16.347493
region_2                 61.136715
taster_name              20.192197
taster_twitter_handle    24.015357
title                     0.000000
variety                   0.000769
winery                    0.000000
dtype: float64

In [4]:
print("mean : " , df['price'].mean())
print("median : " , df['price'].median())
print("variance : " , df['price'].std() ** 2)
print("Standard deviastion : " , df['price'].std())
print("Min : " , df['price'].min())
print("Max : " , df['price'].max())


mean :  35.363389129985535
median :  25.0
variance :  1682.822342407928
Standard deviastion :  41.02221766808723
Min :  4.0
Max :  3300.0


In [5]:
def analyze_column(target_col):
    std = target_col.std()
    skewness = target_col.skew() if std > 0 else np.nan

    if target_col.count() == 0:
        return pd.Series({
            "mean": np.nan,
            "median": np.nan,
            "variance": np.nan,
            "standard_deviation": np.nan,
            "Min": np.nan,
            "Max": np.nan,
            "Q1": np.nan,
            "Q3": np.nan,
            "skewness": np.nan
        })
    else : 
        statistic_data = {
            "mean": target_col.mean(),
            "median": target_col.median(),
            "variance": target_col.var(),
            "standard_deviation": std,
            "Min": target_col.min(),
            "Max": target_col.max(),
            "Q1": target_col.quantile(0.25),
            "Q3": target_col.quantile(0.75),
            "skewness": skewness
        }

        return pd.Series(statistic_data)

In [6]:
def analyze_dataset(df , cols):
    results = pd.DataFrame(columns=cols)

    for col in cols :
        results[col] = analyze_column(df[col])

    return results 

analyze_dataset(df,["price" , "points"])

        

,price,points
mean,35.363389,88.447138
median,25.000000,88.000000
variance,1682.822342,9.239960
standard_deviation,41.022218,3.039730
Min,4.000000,80.000000
Max,3300.000000,100.000000
Q1,17.000000,86.000000
Q3,42.000000,91.000000
skewness,18.000957,0.045921


In [7]:
test_corr = df[['price','points']]
test_corr.corr(method="pearson")

,price,points
price,1.000000,0.416167
points,0.416167,1.000000


In [8]:
def analyze_correlation(df,cols):

    selected_cols = df[cols]
    corr_res = selected_cols.corr(method="pearson")

    return corr_res
analyze_correlation(df,["price","points"])

,price,points
price,1.000000,0.416167
points,0.416167,1.000000


In [9]:
def detect_outliers(col):
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1

    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    outliers = col[(col < lower) | (col > upper)]

    valid_count = col.count()
    outlier_count = outliers.count()

    outlier_percentage = (
        outlier_count / valid_count * 100
        if valid_count > 0
        else np.nan
    )

    return {
        "IQR": iqr,
        "Upper_bound": upper,
        "Lower_bound": lower,
        "outlier_count": outlier_count,
        "outlier_percentage": outlier_percentage
    }

In [10]:
def generate_report(df, cols):
    result = {}

    # 1. Descriptive Statistics
    result["statistics"] = analyze_dataset(df, cols).T

    # 2. Correlation Analysis
    result["correlation"] = analyze_correlation(df, cols)

    # 3. Outlier Detection
    outliers = {}

    for col in cols:
        outliers[col] = detect_outliers(df[col])

    result["outliers"] = pd.DataFrame.from_dict(
        outliers, orient="index"
    )

    return result

In [11]:
report = generate_report(df, ["price", "points"])

In [12]:
display(report["statistics"])
display(report["correlation"])
display(report["outliers"])

,mean,median,variance,standard_deviation,Min,Max,Q1,Q3,skewness
price,35.363389,25.0,1682.822342,41.022218,4.0,3300.0,17.0,42.0,18.000957
points,88.447138,88.0,9.239960,3.039730,80.0,100.0,86.0,91.0,0.045921


,price,points
price,1.000000,0.416167
points,0.416167,1.000000


,IQR,Upper_bound,Lower_bound,outlier_count,outlier_percentage
price,25.0,79.5,-20.5,7241,5.985534
points,5.0,98.5,78.5,52,0.040009


In [13]:
# Test 1: Descriptive Statistics

test_data = pd.Series([10, 20, 30, 40, 50])

result = analyze_column(test_data)

assert result["mean"] == 30
assert result["median"] == 30
assert result["Min"] == 10
assert result["Max"] == 50

print("Test 1 PASSED: Descriptive Statistics")

Test 1 PASSED: Descriptive Statistics


In [14]:
# Test 2: Correlation Analysis

test_df = pd.DataFrame({
    "x": [1, 2, 3, 4, 5],
    "y": [2, 4, 6, 8, 10]
})

result = analyze_correlation(test_df, ["x", "y"])

assert abs(result.loc["x", "y"] - 1.0) < 1e-10

print("Test 2 PASSED: Correlation Analysis")

Test 2 PASSED: Correlation Analysis


In [15]:
# Test 3: Outlier Detection

test_data = pd.Series([10, 11, 12, 13, 14, 15, 100])

result = detect_outliers(test_data)

assert result["outlier_count"] == 1
assert abs(result["outlier_percentage"] - (100 / 7)) < 1e-10
assert result["Lower_bound"] < 10
assert result["Upper_bound"] < 100

print("Test 3 PASSED: Outlier Detection")

Test 3 PASSED: Outlier Detection


In [16]:
# Test 4: Missing Values

import numpy as np

test_data = pd.Series([10, 20, np.nan, 30, 40])

stats = analyze_column(test_data)
outliers = detect_outliers(test_data)

assert stats["mean"] == 25
assert stats["median"] == 25
assert outliers["outlier_count"] == 0
assert outliers["outlier_percentage"] == 0

print("Test 4 PASSED: Missing Values")

Test 4 PASSED: Missing Values


In [17]:
# Test 5: Constant Values

test_data = pd.Series([10, 10, 10, 10, 10])

stats = analyze_column(test_data)
outliers = detect_outliers(test_data)

assert stats["mean"] == 10
assert stats["median"] == 10
assert stats["standard_deviation"] == 0
assert pd.isna(stats["skewness"])

assert outliers["IQR"] == 0
assert outliers["outlier_count"] == 0
assert outliers["outlier_percentage"] == 0

print("Test 5 PASSED: Constant Values")

Test 5 PASSED: Constant Values


In [18]:
# Test 6: All Missing Values

test_data = pd.Series([np.nan, np.nan, np.nan])

stats = analyze_column(test_data)
outliers = detect_outliers(test_data)

assert pd.isna(stats["mean"])
assert pd.isna(stats["median"])
assert pd.isna(stats["skewness"])

assert outliers["outlier_count"] == 0
assert pd.isna(outliers["outlier_percentage"])

print("Test 6 PASSED: All Missing Values")

Test 6 PASSED: All Missing Values
